# ExoData Exoplanet Clustering - EDA

Notebook renderizable para revisar el analisis exploratorio del dataset `PSCompPars` del NASA Exoplanet Archive. El PDF del concurso enfoca el analisis en variables fisicas, orbitales, estelares y de habitabilidad, no en las 320 columnas completas como matriz directa de clustering.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from feature_config import CONTEST_NUMERIC_COLUMNS, CLUSTERING_FEATURE_SETS, RADIUS_BINS, RADIUS_LABELS

pio.renderers.default = 'notebook_connected'
REPORTS = PROJECT_ROOT / 'reports'
csv_path = sorted(PROJECT_ROOT.glob('PSCompPars_*.csv'))[-1]
display(Markdown(f'**CSV:** `{csv_path.name}`'))

**CSV:** `PSCompPars_2026.04.25_14.43.08.csv`

In [2]:
df = pd.read_csv(csv_path, comment='#', low_memory=False)
missing = pd.read_csv(REPORTS / 'missingness_all_columns.csv')
key_stats = pd.read_csv(REPORTS / 'key_variable_stats.csv')
coverage = pd.read_csv(REPORTS / 'clustering_feature_coverage.csv')
corr_core = pd.read_csv(REPORTS / 'correlation_spearman_core.csv', index_col=0)
strong_corr = pd.read_csv(REPORTS / 'strong_correlations.csv')

summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'numeric_columns', 'columns_50pct_missing', 'columns_80pct_missing'],
    'value': [
        df.shape[0],
        df.shape[1],
        df.select_dtypes(include='number').shape[1],
        int((missing['missing_pct'] >= 50).sum()),
        int((missing['missing_pct'] >= 80).sum()),
    ],
})
summary

,metric,value
0,rows,6273
1,columns,320
2,numeric_columns,240
3,columns_50pct_missing,82
4,columns_80pct_missing,46


## Decision de variables

Para clustering conviene empezar con variables interpretables del PDF. Las columnas completas se usan para auditoria de calidad, nulos y sesgos, pero no como input directo del modelo.

In [3]:
coverage[['feature_set', 'n_features', 'complete_rows', 'complete_pct', 'columns']]

,feature_set,n_features,complete_rows,complete_pct,columns
0,pdf_core_no_mass_density,7,4960,79.069,"pl_rade, pl_orbper, pl_orbsmax, pl_orbeccen, s..."
1,pdf_core_with_mass_density,9,4892,77.985,"pl_rade, pl_bmasse, pl_dens, pl_orbper, pl_orb..."
2,habitability,7,4132,65.870,"pl_rade, pl_orbper, pl_orbsmax, st_teff, pl_in..."
3,wide_physical,11,3848,61.342,"pl_rade, pl_bmasse, pl_dens, pl_orbper, pl_orb..."


In [4]:
fig = px.bar(
    coverage,
    x='feature_set',
    y='complete_pct',
    hover_data=['complete_rows', 'n_features', 'columns'],
    title='Cobertura de conjuntos de variables para clustering',
    labels={'complete_pct': '% filas completas', 'feature_set': 'conjunto'},
    height=480,
)
fig.update_layout(xaxis_tickangle=-25)
fig.show()

## Nulos y rangos de variables clave

In [5]:
key_stats[['column', 'role', 'non_null', 'missing_pct', 'min', 'p25', 'median', 'p75', 'p95', 'p99', 'max']]

,column,role,non_null,missing_pct,min,p25,median,p75,p95,p99,max
0,pl_insol,insolacion,4389,30.033,0.000300,24.1000,99.600000,375.550000,1768.757200,4354.499040,4.490000e+04
1,pl_eqt,temperatura equilibrio,4664,25.650,34.000000,569.0000,822.000000,1162.020000,1788.700000,2307.030000,4.050000e+03
2,pl_orbeccen,excentricidad,5210,16.946,0.000000,0.0000,0.000000,0.090000,0.410000,0.730000,9.500000e-01
3,st_met,metalicidad estrella,5609,10.585,-1.000000,-0.0800,0.020000,0.130000,0.320000,0.410000,7.900000e-01
4,pl_orbsmax,semi eje mayor,5834,6.998,0.004400,0.0523,0.102085,0.307750,4.064900,67.670000,1.900000e+04
5,pl_orbper,periodo orbital,5929,5.484,0.090706,4.3050,10.681572,38.021000,1527.000000,13689.235949,4.020000e+08
6,st_teff,temperatura estrella,5959,5.006,415.000000,4895.9850,5542.000000,5894.500000,6340.000000,7268.260000,5.700000e+04
7,pl_dens,densidad planeta,6123,2.391,0.005100,1.3000,2.530000,4.540000,9.740000,27.456000,2.000000e+03
8,pl_rade,radio planeta,6212,0.972,0.309800,1.8400,2.840000,11.890682,14.300000,18.260806,8.720587e+01
9,pl_bmasse,masa planeta,6231,0.670,0.020000,4.2700,9.190000,181.163100,2511.162244,6477.290500,9.534852e+03


In [6]:
key_missing = missing[missing['column'].isin(CONTEST_NUMERIC_COLUMNS)].copy()
fig = px.bar(
    key_missing.sort_values('missing_pct'),
    x='missing_pct',
    y='column',
    orientation='h',
    color='missing_pct',
    color_continuous_scale='Tealrose',
    title='Nulos en variables clave del concurso',
    labels={'missing_pct': '% nulos', 'column': 'variable'},
    height=520,
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## Distribuciones

Muchas variables planetarias son muy sesgadas; por eso se revisan tambien en escala `log10`.

In [7]:
plot_cols = [c for c in CONTEST_NUMERIC_COLUMNS if c in df.columns]
long = df[plot_cols].melt(var_name='variable', value_name='value').dropna()
fig = px.histogram(
    long,
    x='value',
    facet_col='variable',
    facet_col_wrap=3,
    nbins=60,
    title='Distribuciones crudas de variables clave',
    height=900,
)
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show()

In [8]:
log_cols = ['pl_rade', 'pl_bmasse', 'pl_dens', 'pl_orbper', 'pl_orbsmax', 'pl_insol', 'pl_eqt']
log_cols = [c for c in log_cols if c in df.columns]
log_df = df[log_cols].where(df[log_cols] > 0).apply(np.log10)
log_long = log_df.melt(var_name='variable', value_name='log10_value').dropna()
fig = px.histogram(
    log_long,
    x='log10_value',
    facet_col='variable',
    facet_col_wrap=3,
    nbins=60,
    title='Distribuciones en escala log10',
    height=780,
)
fig.update_xaxes(matches=None)
fig.update_yaxes(matches=None)
fig.show()

## Correlaciones

In [9]:
fig = px.imshow(
    corr_core,
    zmin=-1,
    zmax=1,
    color_continuous_scale='RdBu_r',
    title='Correlacion Spearman: variables clave',
    height=680,
)
fig.update_xaxes(tickangle=-45)
fig.show()

In [10]:
strong_corr.head(20)

,feature_a,feature_b,spearman
0,pl_bmasse,pl_bmassj,0.999996
1,pl_rade,pl_radj,0.999977
2,sy_w1mag,sy_w2mag,0.999637
3,sy_hmag,sy_kmag,0.999501
4,sy_kmag,sy_w1mag,0.999186
5,sy_kmag,sy_w2mag,0.998740
6,sy_hmag,sy_w1mag,0.998636
7,sy_jmag,sy_hmag,0.998066
8,sy_hmag,sy_w2mag,0.997864
9,sy_dist,sy_plx,-0.997331


## Relaciones fisicas y sesgos de descubrimiento

In [11]:
plot_df = df.dropna(subset=['pl_orbper', 'pl_rade']).copy()
plot_df = plot_df[(plot_df['pl_orbper'] > 0) & (plot_df['pl_rade'] > 0)]
fig = px.scatter(
    plot_df,
    x='pl_orbper',
    y='pl_rade',
    color='discoverymethod',
    hover_name='pl_name',
    hover_data=['hostname', 'disc_year'],
    log_x=True,
    log_y=True,
    title='Radio vs periodo orbital por metodo de descubrimiento',
    labels={'pl_orbper': 'periodo orbital (dias)', 'pl_rade': 'radio planeta (R_earth)'},
    height=620,
)
fig.show()

In [12]:
radius_df = df.copy()
radius_df['radius_class'] = pd.cut(radius_df['pl_rade'], bins=RADIUS_BINS, labels=RADIUS_LABELS)
counts = radius_df['radius_class'].value_counts(dropna=False).rename_axis('radius_class').reset_index(name='count')
counts['radius_class'] = counts['radius_class'].astype(str)
fig = px.bar(
    counts,
    x='radius_class',
    y='count',
    title='Clasificacion por radio sugerida en la guia',
    labels={'radius_class': 'clase por radio', 'count': 'planetas'},
    height=480,
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()